In [9]:
import json
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import plotly.express as px
import requests

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    pass

API_KEY = 'paste_API_KEY' 
if not API_KEY:
    raise RuntimeError(
        "TICKETMASTER_API_KEY not set. Set it in your terminal environment or .env file."
    )

BASE_URL = "https://app.ticketmaster.com/discovery/v2/events.json"
PAGE_SIZE = 200
RECORD_CAP = 1000
MONTHS_AHEAD = 12
TOP_N_CITIES = 10
ONS_POPULATION_DATA = {
    "London": 8908000,
    "Birmingham": 1144000,
    "Leeds": 812000,
    "Glasgow": 635000,
    "Sheffield": 584000,
    "Manchester": 553000,
    "Bradford": 546000,
    "Edinburgh": 526000,
    "Liverpool": 496000,
    "Bristol": 472000,
    "Cardiff": 372000,
    "Leicester": 368000,
    "Coventry": 345000,
    "Belfast": 345000,
    "Nottingham": 323000,
    "Newcastle": 300000,
    "Brighton": 277000,
    "Brighton and Hove": 277000,
    "Derby": 261000,
    "Hull": 260000,
    "Stoke-on-Trent": 258000,
    "Southampton": 253000,
    "Milton Keynes": 230000,
    "York": 210000,
    "Portsmouth": 208000,
    "Aberdeen": 200000,
    "Plymouth": 264000,
    "Reading": 174000,
    "Oxford": 162000,
    "Cambridge": 145000,
    "Norwich": 144000,
}


def _monthly_windows(months_ahead):
    """Generates monthly date ranges to stay under Ticketmaster's 1,000-record limit."""
    start = datetime.utcnow().replace(hour=0, minute=0, second=0, microsecond=0)
    for i in range(months_ahead):
        window_start = start + timedelta(days=30 * i)
        window_end = window_start + timedelta(days=30)
        yield (
            window_start.strftime("%Y-%m-%dT00:00:00Z"),
            window_end.strftime("%Y-%m-%dT00:00:00Z"),
        )


def fetch_raw_events(genre_name):
    """Queries Ticketmaster live, tracks API call count, and saves raw JSON safely to disk."""
    os.makedirs("raw_data", exist_ok=True)
    all_raw_events = []
    api_call_count = 0 

    print(f"\n=== STARTING TICKETMASTER INGESTION FOR GENRE: '{genre_name}' ===")

    for window_start, window_end in _monthly_windows(MONTHS_AHEAD):
        page = 0
        while True:
            # Guardrail: Check for query scope cap
            if page * PAGE_SIZE >= RECORD_CAP:
                print(
                    f"⚠️ [Truncation Warning] Date window {window_start[:7]} hit the 1,000-record cap."
                )
                break

            params = {
                "apikey": API_KEY,
                "countryCode": "GB",
                "segmentName": "Music",
                "classificationName": genre_name,
                "startDateTime": window_start,
                "endDateTime": window_end,
                "size": PAGE_SIZE,
                "page": page,
            }

            response = requests.get(BASE_URL, params=params)
            api_call_count += 1  # Increment on each request execution
            time.sleep(0.2)      # Rate limiter (~5 requests/sec)

            if response.status_code != 200:
                print(
                    f"Error fetching {window_start[:7]} (Page {page}): HTTP {response.status_code}"
                )
                break

            data = response.json()
            events = data.get("_embedded", {}).get("events", [])
            all_raw_events.extend(events)

            # Pagination Logging Guardrail
            page_info = data.get("page", {})
            total_pages = page_info.get("totalPages", 1)
            total_elements = page_info.get("totalElements", 0)

            print(
                f"Window: {window_start[:7]} | Page {page + 1}/{total_pages} | "
                f"Fetched: {len(events)} | Available: {total_elements}"
            )

            page += 1
            if page >= total_pages:
                break

    # Guardrail: Store raw response payload to disk (sanitizing slashes/spaces in genre name)
    clean_filename_genre = genre_name.lower().replace(" ", "_").replace("/", "_")
    raw_file = os.path.join(
        "raw_data", f"raw_{clean_filename_genre}_events.json"
    )

    with open(raw_file, "w", encoding="utf-8") as f:
        json.dump(all_raw_events, f, ensure_ascii=False, indent=2)

    print(
        f"\n✅ Ingestion complete. Saved {len(all_raw_events)} raw events across {api_call_count} API calls to '{raw_file}'."
    )
    return all_raw_events


def parse_and_clean_data(raw_events):
    """Extracts ONLY minimal required fields: event_id, attraction_id, city, genre."""
    parsed = []

    for ev in raw_events:
        classifications = ev.get("classifications", [{}])[0]
        genre = classifications.get("genre", {}).get("name")

        venues = ev.get("_embedded", {}).get("venues", [{}])
        city = venues[0].get("city", {}).get("name") if venues else None

        attractions = ev.get("_embedded", {}).get("attractions", [{}])
        attraction_id = attractions[0].get("id") if attractions else None

        parsed.append(
            {
                "event_id": ev.get("id"),
                "attraction_id": attraction_id,
                "city": city,
                "genre": genre,
            }
        )

    df = pd.DataFrame(parsed)

    if "city" in df.columns:
        df["city"] = df["city"].astype(str).str.strip()

    print("\n--- NULL RATE REPORT ---")
    for col in ["attraction_id", "city", "genre"]:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100 if len(df) > 0 else 0
        print(f"Column '{col}': {null_count} missing ({null_pct:.1f}%)")

    before_count = len(df)
    df_cleaned = df.dropna(subset=["attraction_id", "city", "genre"]).copy()
    print(
        f"Applied NULL Strategy 'Exclude': Dropped {before_count - len(df_cleaned)} incomplete records."
    )

    return df_cleaned


def rank_top_cities(cleaned_df, top_n=TOP_N_CITIES):
    """Deduplicates acts by attraction_id per city FIRST, then ranks top N markets."""
    # Guardrail: Deduplicate acts per city
    deduped_df = cleaned_df.drop_duplicates(subset=["city", "attraction_id"])

    city_summary = (
        deduped_df.groupby("city")
        .agg(
            unique_acts=("attraction_id", "nunique"),
            total_events=("event_id", "count"),
        )
        .reset_index()
        .sort_values(by="unique_acts", ascending=False)
    )

    return city_summary.head(top_n)


def compute_market_saturation(city_summary):
    """Maps population reference, flags missing cities, and calculates acts per 100k baseline."""
    summary = city_summary.copy()
    summary["population_ons"] = summary["city"].map(ONS_POPULATION_DATA)

    # Flag missing cities from population reference
    missing = summary[summary["population_ons"].isnull()]["city"].tolist()
    if missing:
        print(
            f"\n⚠️ [FLAGGED ISSUE] Missing population reference for: {missing}. "
            "Falling back to raw unique_acts count as proxy."
        )

    summary["acts_per_100k"] = (
        summary["unique_acts"] / summary["population_ons"]
    ) * 100000
    summary["acts_per_100k"] = summary["acts_per_100k"].fillna(
        summary["unique_acts"]
    )
    summary = summary.sort_values(by="acts_per_100k", ascending=False)

    return summary


def plot_saturation(saturation_df, genre_name):
    """Displays an interactive Plotly bar chart with a Red-White gradient and mean line."""
    mean_saturation = saturation_df["acts_per_100k"].mean()

    custom_color_scale = [
        [0.0, "#C8B8F0"],  # Light: Pale lavender (Undersaturated)
        [0.5, "#6B35C8"],  # Secondary: Mid purple (Middle)
        [1.0, "#1A3A8F"], 
    ]
    fig = px.bar(
        saturation_df,
        x="city",
        y="acts_per_100k",
        text_auto=".1f",
        title=f"Market Saturation: {genre_name.title()} (Acts per 100k Residents)",
        labels={
            "city": "City / Market",
            "acts_per_100k": "Unique Acts per 100k Residents",
        },
        color="acts_per_100k",
        color_continuous_scale=brand_color_scale,
    )

    # Accent Teal/Cyan (#00B4C8) for the Mean Baseline Line
    fig.add_hline(
        y=mean_saturation,
        line_dash="dash",
        line_color="#00B4C8",  # Accent
        line_width=2,
        annotation_text=f"Market Mean Baseline ({mean_saturation:.1f})",
        annotation_position="top right",
        annotation_font=dict(color="#0D1F5C", size=12, family="Arial"),
    )

    # Layout styling using Neutral background and Dark navy text
    fig.update_layout(
        xaxis_title="UK City",
        yaxis_title="Acts per 100,000",
        plot_bgcolor="#F4F4F6",   # Neutral (Off-white chart background)
        paper_bgcolor="#FFFFFF",  # Canvas background
        font=dict(color="#0D1F5C"),  # Dark (Near-black navy text)
    )

    fig.show()
if __name__ == "__main__":
    user_genre = input(
        "Enter genre to analyze (e.g. Rock, Pop, Dance/Electronic, Hip-Hop/Rap): "
    ).strip()
    if not user_genre:
        user_genre = "Rock"

    raw_events = fetch_raw_events(genre_name=user_genre)

    cleaned_df = parse_and_clean_data(raw_events)
    cleaned_df.to_csv("cleaned_events.csv", index=False)

    top_cities = rank_top_cities(cleaned_df)
    saturation_df = compute_market_saturation(top_cities)
    saturation_df.to_csv("market_saturation.csv", index=False)

    print(
        f"\n=== MARKET SATURATION SUMMARY ({user_genre.upper()} - Top 10 Cities) ==="
    )
    print(saturation_df.to_string(index=False))

    plot_saturation(saturation_df, genre_name=user_genre)

RuntimeError: TICKETMASTER_API_KEY not set. Set it in your terminal environment or .env file.